# HateGuard 

Toxic Comment Detection
and Video Analysis

In [ ]:
import os
# 1. cloning the repository
!rm -rf hate-comment-dectection
!git clone https://github.com/ATANU28-bit/hate-comment-dectection.git
%cd hate-comment-dectection

In [ ]:
# 2. Installing dependencies
!sudo apt update && sudo apt install ffmpeg -y
!pip install -r requirements.txt

# Upgrade libraries to latest versions
!pip install --upgrade youtube-comment-downloader pytubefix

# Installing UI dependencies
%cd ui
!npm install
%cd ..

### 3. One-Time YouTube Authentication
Run the cell below. It will show a link and a code. 
1. Click the link (google.com/device).
2. Enter the code shown in the output.
3. This authorizes audio downloads so you are not detected as a bot.

In [ ]:
from pytubefix import YouTube
print("Starting Authentication...")
# We trigger the auth flow here interactively before starting the server
try:
    yt = YouTube('https://youtube.com/watch?v=jNQXAC9IVRw', use_oauth=True, allow_oauth_cache=True)
    print("\nSuccess! You are now authorized.")
except Exception as e:
    print(f"Authentication error: {e}")

In [ ]:
import subprocess
import time
import sys
import threading
from google.colab.output import eval_js

# Setting UI to route queries via Vite's proxy path "/api"
with open("ui/.env", "w", encoding="utf-8") as f:
    f.write("VITE_API_URL=/api\n")

print("Starting Backend on port 8000...")
def stream_logs(proc, prefix=""):
    for line in iter(proc.stdout.readline, b''):
        sys.stdout.write(f"{prefix}{line.decode('utf-8')}")
        sys.stdout.flush()

backend = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "src.api:app", "--host", "127.0.0.1", "--port", "8000"], 
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
threading.Thread(target=stream_logs, args=(backend, "[BACKEND] "), daemon=True).start()

print("Starting Frontend Vite proxy on port 5173...")
frontend = subprocess.Popen(
    ["npm", "run", "dev", "--prefix", "ui", "--", "--host", "127.0.0.1", "--port", "5173"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
threading.Thread(target=stream_logs, args=(frontend, "[UI] "), daemon=True).start()

time.sleep(5)
proxy_url = eval_js("google.colab.kernel.proxyPort(5173)")
print(f"\n TO OPEN APP: {proxy_url}\n")
try:
    frontend.wait()
except KeyboardInterrupt:
    backend.terminate()
    frontend.terminate()
